# 06 Metrics - Full Fixed Version

File này đánh giá mô hình gợi ý với 2 phần:

1. **Đánh giá tổng thể**
   - Without Taxonomy
   - With Taxonomy

2. **Đánh giá riêng taxonomy-match cases**
   - Giữ toàn bộ ranking.
   - Dùng `taxonomy_match` làm nhãn `relevant`.
   - Không lọc bỏ negative samples.

Metrics:
- Precision@10, Recall@10, NDCG@10
- Precision@20, Recall@20, NDCG@20
- AUC
- Mean Rank Relevant
- MRR


In [28]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding")
DATA_DIR = BASE_DIR / "data_outputs"

RANKING_PARQUET = DATA_DIR / "13_candidate_job_hybrid_ranking.parquet"
RANKING_EXCEL = DATA_DIR / "13_candidate_job_hybrid_ranking.xlsx"

OUTPUT_CANDIDATE_METRICS = DATA_DIR / "15_candidate_level_metrics.xlsx"
OUTPUT_SUMMARY = DATA_DIR / "16_metrics_summary.xlsx"

OUTPUT_TAXONOMY_CASE_SUMMARY = DATA_DIR / "17_taxonomy_case_metrics_summary.xlsx"
OUTPUT_TAXONOMY_CASE_DETAIL = DATA_DIR / "17_taxonomy_case_candidate_metrics.xlsx"

print("RANKING_PARQUET exists:", RANKING_PARQUET.exists(), RANKING_PARQUET)
print("RANKING_EXCEL exists:", RANKING_EXCEL.exists(), RANKING_EXCEL)


RANKING_PARQUET exists: True /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.parquet
RANKING_EXCEL exists: True /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.xlsx


In [29]:
# ===== READ RANKING FILE =====
if RANKING_PARQUET.exists():
    df = pd.read_parquet(RANKING_PARQUET)
elif RANKING_EXCEL.exists():
    df = pd.read_excel(RANKING_EXCEL)
else:
    raise FileNotFoundError("Không tìm thấy 13_candidate_job_hybrid_ranking.parquet hoặc .xlsx")

print("df:", df.shape)
print("Candidates:", df["candidate_id"].nunique())
print("Jobs:", df["job_id"].nunique())
print("Min jobs per candidate:", df.groupby("candidate_id")["job_id"].count().min())
print("Max jobs per candidate:", df.groupby("candidate_id")["job_id"].count().max())

df.head()


df: (400, 20)
Candidates: 20
Jobs: 53
Min jobs per candidate: 20
Max jobs per candidate: 20


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,job_title,match_explanation,semantic_similarity,semantic_rank,semantic_score_norm,baseline_score_norm,hybrid_no_taxonomy_score,hybrid_taxonomy_score,hybrid_score,rank_taxonomy,rank_no_taxonomy,hybrid_rank,score_diff,rank_diff
0,C001,J042,1.0000,1.0,1,1.0000,Automation Test Engineer,skill overlap=1.0; group similarity=1.0; same ...,0.843510,18,0.921755,1.0000,1.0000,1.00000,1.00000,1,1,1,0.00000,0
1,C001,J048,0.5000,1.0,1,0.6750,Test Automation Developer,skill overlap=0.5; group similarity=1.0; same ...,0.842319,19,0.921160,0.6750,0.5000,0.75000,0.75000,2,2,2,0.25000,0
2,C001,J005,0.3333,1.0,1,0.5667,React Frontend Developer,skill overlap=0.3333; group similarity=1.0; sa...,0.906228,1,0.953114,0.5667,0.3333,0.66665,0.66665,3,3,3,0.33335,0
3,C001,J006,0.3333,1.0,1,0.5667,Svelte Frontend Developer,skill overlap=0.3333; group similarity=1.0; sa...,0.880208,5,0.940104,0.5667,0.3333,0.66665,0.66665,4,4,4,0.33335,0
4,C001,J007,0.3333,1.0,1,0.5667,Web Component Engineer,skill overlap=0.3333; group similarity=1.0; sa...,0.869554,8,0.934777,0.5667,0.3333,0.66665,0.66665,5,5,5,0.33335,0


In [30]:
# ===== CHECK REQUIRED COLUMNS =====
required_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

numeric_cols = [
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df.head()


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,job_title,match_explanation,semantic_similarity,semantic_rank,semantic_score_norm,baseline_score_norm,hybrid_no_taxonomy_score,hybrid_taxonomy_score,hybrid_score,rank_taxonomy,rank_no_taxonomy,hybrid_rank,score_diff,rank_diff
0,C001,J042,1.0000,1.0,1,1.0000,Automation Test Engineer,skill overlap=1.0; group similarity=1.0; same ...,0.843510,18,0.921755,1.0000,1.0000,1.00000,1.00000,1,1,1,0.00000,0
1,C001,J048,0.5000,1.0,1,0.6750,Test Automation Developer,skill overlap=0.5; group similarity=1.0; same ...,0.842319,19,0.921160,0.6750,0.5000,0.75000,0.75000,2,2,2,0.25000,0
2,C001,J005,0.3333,1.0,1,0.5667,React Frontend Developer,skill overlap=0.3333; group similarity=1.0; sa...,0.906228,1,0.953114,0.5667,0.3333,0.66665,0.66665,3,3,3,0.33335,0
3,C001,J006,0.3333,1.0,1,0.5667,Svelte Frontend Developer,skill overlap=0.3333; group similarity=1.0; sa...,0.880208,5,0.940104,0.5667,0.3333,0.66665,0.66665,4,4,4,0.33335,0
4,C001,J007,0.3333,1.0,1,0.5667,Web Component Engineer,skill overlap=0.3333; group similarity=1.0; sa...,0.869554,8,0.934777,0.5667,0.3333,0.66665,0.66665,5,5,5,0.33335,0


In [31]:
# ===== TAXONOMY-AWARE GROUND TRUTH FOR OVERALL EVALUATION =====
DIRECT_SKILL_THRESHOLD = 0.50
TAXONOMY_GROUP_THRESHOLD = 0.50

df["direct_match"] = (
    df["skill_overlap_score"] >= DIRECT_SKILL_THRESHOLD
).astype(int)

df["taxonomy_match"] = (
    (df["skill_overlap_score"] < DIRECT_SKILL_THRESHOLD)
    & (df["group_similarity_score"] >= TAXONOMY_GROUP_THRESHOLD)
    & (df["dominant_group_score"] >= 1)
).astype(int)

df["relevant"] = (
    (df["direct_match"] == 1)
    | (df["taxonomy_match"] == 1)
).astype(int)

print("Relevant distribution:")
print(df["relevant"].value_counts(dropna=False))

print("\\nMatch type summary:")
print("Direct matches:", int(df["direct_match"].sum()))
print("Taxonomy matches:", int(df["taxonomy_match"].sum()))
print("Total relevant:", int(df["relevant"].sum()))

df[[
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "direct_match",
    "taxonomy_match",
    "relevant",
]].head(20)


Relevant distribution:
relevant
1    236
0    164
Name: count, dtype: int64
\nMatch type summary:
Direct matches: 116
Taxonomy matches: 120
Total relevant: 236


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,direct_match,taxonomy_match,relevant
0,C001,J042,1.0000,1.0,1,1,0,1
1,C001,J048,0.5000,1.0,1,1,0,1
2,C001,J005,0.3333,1.0,1,0,1,1
3,C001,J006,0.3333,1.0,1,0,1,1
4,C001,J007,0.3333,1.0,1,0,1,1
5,C001,J001,0.0000,1.0,1,0,1,1
6,C001,J002,0.0000,1.0,1,0,1,1
7,C001,J003,0.0000,1.0,1,0,1,1
8,C001,J004,0.0000,1.0,1,0,1,1
9,C001,J008,0.0000,1.0,1,0,1,1


In [32]:
# ===== SANITY CHECK =====
job_counts = df.groupby("candidate_id")["job_id"].count()
relevant_counts = df.groupby("candidate_id")["relevant"].sum()
taxonomy_counts = df.groupby("candidate_id")["taxonomy_match"].sum()

summary_check = pd.DataFrame({
    "n_jobs": job_counts,
    "n_relevant": relevant_counts,
    "n_taxonomy_match": taxonomy_counts,
}).reset_index()

print(summary_check.describe())

if summary_check["n_jobs"].min() < 20:
    print("WARNING: Có candidate có ít hơn 20 jobs. Metrics @20 vẫn tính được, nhưng chưa thật sự đẹp về ý nghĩa.")

summary_check.head(30)


       n_jobs  n_relevant  n_taxonomy_match
count    20.0   20.000000         20.000000
mean     20.0   11.800000          6.000000
std       0.0    7.990784          7.033753
min      20.0    0.000000          0.000000
25%      20.0    4.000000          0.000000
50%      20.0   12.500000          3.000000
75%      20.0   20.000000         12.500000
max      20.0   20.000000         19.000000


,candidate_id,n_jobs,n_relevant,n_taxonomy_match
0,C001,20,20,18
1,C002,20,20,19
2,C003,20,13,0
3,C004,20,4,0
4,C005,20,0,0
5,C006,20,0,0
6,C007,20,7,4
7,C008,20,3,2
8,C009,20,12,0
9,C010,20,20,8


In [33]:
# ===== METRIC FUNCTIONS =====
K_VALUES = [10, 20]

def precision_at_k(group, rank_col, k):
    top_k = group.sort_values(rank_col).head(k)
    if len(top_k) == 0:
        return 0.0
    return top_k["relevant"].sum() / k


def recall_at_k(group, rank_col, k):
    total_relevant = group["relevant"].sum()
    if total_relevant == 0:
        return 0.0
    top_k = group.sort_values(rank_col).head(k)
    return top_k["relevant"].sum() / total_relevant


def dcg_at_k(relevances):
    relevances = np.asarray(relevances, dtype=float)
    if relevances.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, relevances.size + 2))
    return float(np.sum(relevances / discounts))


def ndcg_at_k(group, rank_col, k):
    top_k = group.sort_values(rank_col).head(k)
    actual_relevance = top_k["relevant"].values
    dcg = dcg_at_k(actual_relevance)

    ideal_relevance = np.sort(group["relevant"].values)[::-1][:k]
    idcg = dcg_at_k(ideal_relevance)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def safe_auc(data, score_col):
    y_true = data["relevant"].astype(int)
    y_score = data[score_col].astype(float)

    if y_true.nunique() < 2:
        return np.nan

    return roc_auc_score(y_true, y_score)


def mean_rank_of_relevant(group, rank_col):
    relevant_rows = group[group["relevant"] == 1]
    if len(relevant_rows) == 0:
        return np.nan
    return relevant_rows[rank_col].mean()


def mrr(group, rank_col):
    relevant_rows = group[group["relevant"] == 1].sort_values(rank_col)
    if len(relevant_rows) == 0:
        return 0.0
    first_rank = relevant_rows.iloc[0][rank_col]
    if first_rank <= 0:
        return 0.0
    return 1.0 / first_rank


In [34]:
# ===== EVALUATE MODEL =====
def evaluate_model(eval_df, model_name, score_col, rank_col):
    rows = []

    for candidate_id, group in eval_df.groupby("candidate_id"):
        row = {
            "model": model_name,
            "candidate_id": candidate_id,
            "n_jobs": int(group["job_id"].nunique()),
            "n_relevant": int(group["relevant"].sum()),
            "mean_rank_relevant": mean_rank_of_relevant(group, rank_col),
            "mrr": mrr(group, rank_col),
        }

        for k in K_VALUES:
            row[f"precision_at_{k}"] = precision_at_k(group, rank_col, k)
            row[f"recall_at_{k}"] = recall_at_k(group, rank_col, k)
            row[f"ndcg_at_{k}"] = ndcg_at_k(group, rank_col, k)

        rows.append(row)

    candidate_metrics = pd.DataFrame(rows)

    summary = {
        "model": model_name,
        "auc": safe_auc(eval_df, score_col),
        "mean_rank_relevant": candidate_metrics["mean_rank_relevant"].mean(),
        "mean_mrr": candidate_metrics["mrr"].mean(),
    }

    for k in K_VALUES:
        summary[f"mean_precision_at_{k}"] = candidate_metrics[f"precision_at_{k}"].mean()
        summary[f"mean_recall_at_{k}"] = candidate_metrics[f"recall_at_{k}"].mean()
        summary[f"mean_ndcg_at_{k}"] = candidate_metrics[f"ndcg_at_{k}"].mean()

    return candidate_metrics, summary


In [37]:
# ===== OVERALL EVALUATION =====
no_tax_metrics, no_tax_summary = evaluate_model(
    eval_df=df,
    model_name="Without Taxonomy",
    score_col="hybrid_no_taxonomy_score",
    rank_col="rank_no_taxonomy",
)

tax_metrics, tax_summary = evaluate_model(
    eval_df=df,
    model_name="With Taxonomy",
    score_col="hybrid_taxonomy_score",
    rank_col="rank_taxonomy",
)

candidate_level_metrics = pd.concat(
    [no_tax_metrics, tax_metrics],
    ignore_index=True,
)

metrics_summary = pd.DataFrame([
    no_tax_summary,
    tax_summary,
])

overall_display_cols = [
    "model",
    "mean_precision_at_10",
    "mean_recall_at_10",
    "mean_ndcg_at_10",
    "mean_precision_at_20",
    "mean_recall_at_20",
    "mean_ndcg_at_20",
    "auc",
    "mean_rank_relevant",
    "mean_mrr",
]

metrics_summary[overall_display_cols]


,model,mean_precision_at_10,mean_recall_at_10,mean_ndcg_at_10,mean_precision_at_20,mean_recall_at_20,mean_ndcg_at_20,auc,mean_rank_relevant,mean_mrr
0,Without Taxonomy,0.715,0.599825,0.841823,0.59,0.85,0.847821,0.768706,7.516043,0.85
1,With Taxonomy,0.725,0.608916,0.850000,0.59,0.85,0.850000,0.998889,7.441176,0.85


## Taxonomy-match evaluation

Không lọc dataset chỉ còn `taxonomy_match == 1`.  
Cách đúng là giữ full ranking và đổi nhãn `relevant = taxonomy_match`, để các job không phải taxonomy-match vẫn là negative samples.


In [38]:
# ===== TAXONOMY-MATCH EVALUATION ON FULL RANKING =====
taxonomy_full_df = df.copy()

taxonomy_full_df["overall_relevant"] = taxonomy_full_df["relevant"]
taxonomy_full_df["relevant"] = taxonomy_full_df["taxonomy_match"].astype(int)

print("taxonomy_full_df:", taxonomy_full_df.shape)
print("Total taxonomy relevant:", int(taxonomy_full_df["relevant"].sum()))
print("Candidates with taxonomy relevant:", int(taxonomy_full_df.groupby("candidate_id")["relevant"].sum().gt(0).sum()))

taxonomy_full_df[[
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "taxonomy_match",
    "relevant",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
]].head(20)


taxonomy_full_df: (400, 24)
Total taxonomy relevant: 120
Candidates with taxonomy relevant: 12


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,taxonomy_match,relevant,hybrid_no_taxonomy_score,hybrid_taxonomy_score,rank_no_taxonomy,rank_taxonomy
0,C001,J042,1.0000,1.0,1,0,0,1.0000,1.00000,1,1
1,C001,J048,0.5000,1.0,1,0,0,0.5000,0.75000,2,2
2,C001,J005,0.3333,1.0,1,1,1,0.3333,0.66665,3,3
3,C001,J006,0.3333,1.0,1,1,1,0.3333,0.66665,4,4
4,C001,J007,0.3333,1.0,1,1,1,0.3333,0.66665,5,5
5,C001,J001,0.0000,1.0,1,1,1,0.0000,0.50000,6,6
6,C001,J002,0.0000,1.0,1,1,1,0.0000,0.50000,7,7
7,C001,J003,0.0000,1.0,1,1,1,0.0000,0.50000,8,8
8,C001,J004,0.0000,1.0,1,1,1,0.0000,0.50000,9,9
9,C001,J008,0.0000,1.0,1,1,1,0.0000,0.50000,10,10


In [39]:
# ===== RUN TAXONOMY-MATCH EVALUATION =====
taxonomy_case_no_tax_metrics, taxonomy_case_no_tax_summary = evaluate_model(
    eval_df=taxonomy_full_df,
    model_name="Without Taxonomy - Taxonomy Match",
    score_col="hybrid_no_taxonomy_score",
    rank_col="rank_no_taxonomy",
)

taxonomy_case_tax_metrics, taxonomy_case_tax_summary = evaluate_model(
    eval_df=taxonomy_full_df,
    model_name="With Taxonomy - Taxonomy Match",
    score_col="hybrid_taxonomy_score",
    rank_col="rank_taxonomy",
)

taxonomy_case_candidate_metrics = pd.concat(
    [taxonomy_case_no_tax_metrics, taxonomy_case_tax_metrics],
    ignore_index=True,
)

taxonomy_case_summary = pd.DataFrame([
    taxonomy_case_no_tax_summary,
    taxonomy_case_tax_summary,
])

taxonomy_case_summary[overall_display_cols]


,model,mean_precision_at_10,mean_recall_at_10,mean_ndcg_at_10,mean_precision_at_20,mean_recall_at_20,mean_ndcg_at_20,auc,mean_rank_relevant,mean_mrr
0,Without Taxonomy - Taxonomy Match,0.24,0.281971,0.241890,0.3,0.6,0.378692,0.319286,11.333333,0.137497
1,With Taxonomy - Taxonomy Match,0.25,0.296257,0.252102,0.3,0.6,0.382697,0.606161,10.750000,0.141187


In [40]:
# ===== EXPORT ALL RESULTS =====
candidate_level_metrics.to_excel(OUTPUT_CANDIDATE_METRICS, index=False)
metrics_summary.to_excel(OUTPUT_SUMMARY, index=False)

taxonomy_case_candidate_metrics.to_excel(OUTPUT_TAXONOMY_CASE_DETAIL, index=False)
taxonomy_case_summary.to_excel(OUTPUT_TAXONOMY_CASE_SUMMARY, index=False)

print("Saved ->", OUTPUT_CANDIDATE_METRICS)
print("Saved ->", OUTPUT_SUMMARY)
print("Saved ->", OUTPUT_TAXONOMY_CASE_DETAIL)
print("Saved ->", OUTPUT_TAXONOMY_CASE_SUMMARY)

print("\\nOverall summary:")
display(metrics_summary[overall_display_cols])

print("\\nTaxonomy-case summary:")
display(taxonomy_case_summary[overall_display_cols])


Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/15_candidate_level_metrics.xlsx
Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/16_metrics_summary.xlsx
Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/17_taxonomy_case_candidate_metrics.xlsx
Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/17_taxonomy_case_metrics_summary.xlsx
\nOverall summary:


,model,mean_precision_at_10,mean_recall_at_10,mean_ndcg_at_10,mean_precision_at_20,mean_recall_at_20,mean_ndcg_at_20,auc,mean_rank_relevant,mean_mrr
0,Without Taxonomy,0.715,0.599825,0.841823,0.59,0.85,0.847821,0.768706,7.516043,0.85
1,With Taxonomy,0.725,0.608916,0.850000,0.59,0.85,0.850000,0.998889,7.441176,0.85


\nTaxonomy-case summary:


,model,mean_precision_at_10,mean_recall_at_10,mean_ndcg_at_10,mean_precision_at_20,mean_recall_at_20,mean_ndcg_at_20,auc,mean_rank_relevant,mean_mrr
0,Without Taxonomy - Taxonomy Match,0.24,0.281971,0.241890,0.3,0.6,0.378692,0.319286,11.333333,0.137497
1,With Taxonomy - Taxonomy Match,0.25,0.296257,0.252102,0.3,0.6,0.382697,0.606161,10.750000,0.141187
